In [ ]:
"""Shared GROMACS setup for the p16-CDK6 simulations. Runs inside a Kaggle kernel."""
import os, subprocess, sys, shutil, textwrap, time
from pathlib import Path
import urllib.request

WORK = Path('/kaggle/working')
MM = Path('/kaggle/tmp/mm')
ENV = 'md'


def sh(cmd, inp=None, check=True, tail=25, cwd=None, quiet=False):
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, input=inp, text=True, capture_output=True, cwd=cwd, executable='/bin/bash')
    out = r.stdout + r.stderr
    if not quiet or r.returncode:
        print(f'$ {cmd[:150]}  [{time.time() - t0:.0f}s, exit {r.returncode}]')
        print('\n'.join(out.splitlines()[-tail:]))
    if check and r.returncode:
        raise RuntimeError(f'failed: {cmd[:150]}')
    return out


def install():
    """Micromamba env with a CUDA build of GROMACS and the Python model-building tools."""
    MM.mkdir(parents=True, exist_ok=True)
    sh(f'cd {MM} && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba')
    print(sh(f'{MM}/bin/micromamba search -c conda-forge "gromacs>=2023" 2>&1 | tail -14', check=False, quiet=True))
    sh(f'{MM}/bin/micromamba create -y -q -r {MM}/r -n {ENV} -c conda-forge "gromacs=*=*cuda*" python=3.11 pdbfixer openmm numpy', check=False, tail=8)
    gmx = f'{MM}/r/envs/{ENV}/bin.AVX2_256/gmx'
    if not Path(gmx).exists():
        gmx = f'{MM}/r/envs/{ENV}/bin/gmx'
    return gmx


def gmx_ok(gmx):
    if not Path(gmx).exists():
        return False
    v = sh(f'{gmx} --version', check=False, quiet=True)
    print('\n'.join(l for l in v.splitlines() if any(k in l for k in ('GROMACS version', 'GPU support', 'SIMD', 'FFT library'))))
    return 'GPU support:        CUDA' in v or 'CUDA' in [l for l in v.splitlines() if 'GPU support' in l][0]


EM = """integrator=steep
emtol=1000
emstep=0.01
nsteps=50000
cutoff-scheme=Verlet
coulombtype=PME
rcoulomb=1.0
rvdw=1.0
pbc=xyz
"""
BASE = """cutoff-scheme=Verlet
coulombtype=PME
rcoulomb=1.0
vdwtype=cut-off
rvdw=1.0
DispCorr=EnerPres
pbc=xyz
tcoupl=V-rescale
tc-grps=System
tau_t=1.0
ref_t=310
constraints=h-bonds
constraint-algorithm=lincs
"""
NVT = BASE + """integrator=md
define=-DPOSRES
dt=0.002
nsteps=100000
nstenergy=5000
nstlog=5000
pcoupl=no
gen_vel=yes
gen_temp=310
gen_seed={seed}
"""
NPT = BASE + """integrator=md
define=-DPOSRES
refcoord_scaling=com
dt=0.002
nsteps=250000
nstenergy=5000
nstlog=5000
pcoupl=C-rescale
tau_p=2.0
ref_p=1.0
compressibility=4.5e-5
continuation=yes
gen_vel=no
"""
PROD = BASE + """integrator=md
dt=0.004
mass-repartition-factor=3
nsteps={nsteps}
nstxout-compressed=12500
compressed-x-precision=1000
compressed-x-grps=Protein
nstenergy=12500
nstlog=25000
pcoupl=C-rescale
tau_p=5.0
ref_p=1.0
compressibility=4.5e-5
continuation=yes
gen_vel=no
"""


In [ ]:
"""Test only: can gmx_MMPBSA read this GROMACS 2026 setup? Uses a few real frames from the kernel H equilibration, no production data."""
import glob, builtins
_f=open(WORK/"x.log","a",buffering=1)
_p=builtins.print
def print(*a,**k):
    _p(*a,**k); _f.write(" ".join(str(x) for x in a)+chr(10))
MB = f'{MM}/bin/micromamba'
MM.mkdir(parents=True, exist_ok=True)
sh(f'cd {MM} && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba')
out = sh(f'{MB} create -y -r {MM}/r -n gm -c conda-forge "gromacs=2026.*=nompi*" 2>&1 | tail -6', check=False)
print(out)
out = sh(f'{MB} create -y -r {MM}/r -n mm -c conda-forge python=3.11 "ambertools=23.*" mpi4py compilers pip 2>&1 | tail -12', check=False)
print(out)
E = f'{MM}/r/envs/mm'
G = f'{MM}/r/envs/gm'
print(sh(f'ls {G} {G}/bin | head -30; ls {E}/bin | wc -l', check=False))
GB = f'export PATH={G}/bin.AVX2_256:{G}/bin:{E}/bin:$PATH; export AMBERHOME={E}; '
print(sh(GB + f'{E}/bin/python -m pip install gmx_MMPBSA 2>&1 | tail -4; which gmx; gmx --version | head -2; {E}/bin/gmx_MMPBSA --version', check=False))
root = Path(next(p for p in glob.glob('/kaggle/input/**/systems/wt/topol.top', recursive=True))).parent
d = WORK / 'test'; d.mkdir(exist_ok=True)
for f in root.glob('*'):
    if f.suffix in ('.top', '.itp', '.gro', '.tpr', '.cpt'):
        shutil.copy(f, d / f.name)
sh(GB + 'gmx mdrun -s npt.tpr -cpi npt.cpt -deffnm x -nsteps 2000 -noappend -ntmpi 1 -ntomp 4 -nb cpu -pme cpu 2>&1 | tail -2', cwd=d, check=False)
sh(GB + 'gmx grompp -f /dev/null -c npt.gro -p topol.top -o /dev/null 2>&1 | tail -1', cwd=d, check=False)
print(sh('ls', cwd=d, quiet=True))
print('=== groups')
print(sh(GB + 'gmx make_ndx -f npt.gro -o index.ndx', inp='ri 1-294\nri 295-421\nq\n', cwd=d, tail=40))
print(sh('grep "\\[" index.ndx', cwd=d, quiet=True, check=False))
index_text = (d / 'index.ndx').read_text().replace('[ r_1-294 ]', '[ CDK6 ]').replace('[ r_295-421 ]', '[ P16 ]')
(d / 'index.ndx').write_text(index_text)
idx = [l.strip(' []\n') for l in index_text.splitlines() if l.startswith('[')]
assert 'CDK6' in idx and 'P16' in idx, idx
print(idx)
# A few real coordinates, protein atoms only, the same format production writes
# Reassemble the periodic complex before both the reference and trajectory are read.
# H writes whole individual chains in different box images; raw npt.gro is not a binding pose.
sh(GB + 'gmx trjconv -s npt.tpr -f npt.gro -o whole.gro -n index.ndx -pbc cluster', inp='Protein\nProtein\n', cwd=d, tail=8)
sh(GB + 'gmx trjconv -s npt.tpr -f whole.gro -o whole.pdb -n index.ndx', inp='Protein\n', cwd=d, tail=4)
sh(GB + 'gmx trjconv -s npt.tpr -f whole.gro -o f.xtc -n index.ndx', inp='Protein\n', cwd=d, tail=4)
(d / 'mm.in').write_text("""&general
startframe=1, endframe=1, interval=1, verbose=1, PBRadii=2,
/
&gb
igb=5, saltcon=0.150,
/
&decomp
idecomp=2, dec_verbose=0, print_res="within 6",
/
""")
a, b = idx.index('CDK6'), idx.index('P16')
print(sh(GB + f'gmx_MMPBSA -O -i mm.in -cs whole.pdb -ci index.ndx -cg {a} {b} -ct f.xtc -cp topol.top -o FINAL.dat -eo FINAL.csv -deo DEC.csv -nogui', cwd=d, tail=45))
print(sh('ls; tail -25 FINAL.dat; head -12 DEC.csv', cwd=d, check=False, tail=60))
for f in d.glob('*'):
    if f.suffix in ('.cpt', '.tpr', '.xtc', '.trr', '.edr', '.nc') or f.name.startswith('_GMX'):
        f.unlink(missing_ok=True)
import re, math
final_text = (d / 'FINAL.dat').read_text(encoding='utf-8')
match = re.search(r'(?m)^\s*(?:\u0394|DELTA\s*)TOTAL\s+([-+0-9.eE]+)', final_text)
assert match and math.isfinite(float(match.group(1))), 'No finite binding energy result'
print('SMOKE TEST PASS: one real equilibration frame; not a production result')
(WORK / 'done.txt').write_text('MM/GBSA smoke test completed; equilibration frame only, not a production result')
